# LIBS

In [44]:
import numpy as np
import pandas as pd
import re

from datetime import datetime, time, date
from unidecode import unidecode

# IMPORT

In [46]:
df_path = "C:\\Users\\SAMSUNG\\OneDrive\\Documentos\\PROJETO INTEGRADOR 1\\dados\\df_cleaned"
df = pd.read_csv(df_path)
df.head()

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA
0,SANTOS,2022-01-05,2021-11-20 00:00:00,21:00:00,NaN,Comércio e Serviços,VILA MATIAS,Art. 213 - Estupro,1,2022
1,SANTOS,2022-01-09,2022-01-09 00:00:00,00:07:39,NaN,Residência,MARAPE,Estupro de vulneravel (art.217-A),1,2022
2,SANTOS,2022-01-16,2022-01-15 00:00:00,19:30:27,NaN,Residência,MACUCO,Art. 213 - Estupro,1,2022
3,SANTOS,2022-01-11,NaN,NaN,EM HORA INCERTA,Residência,MORRO NOVA CINTRA,Estupro de vulneravel (art.217-A),1,2022
4,SANTOS,2022-02-05,2021-10-17 00:00:00,NaN,DE MADRUGADA,Via Pública,MACUCO,Art. 213 - Estupro,2,2022


In [47]:
# uppercase em todo o dataframe
# uppercase in all dataframe

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.upper()

df.head()

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA
0,SANTOS,2022-01-05,2021-11-20 00:00:00,21:00:00,NaN,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022
1,SANTOS,2022-01-09,2022-01-09 00:00:00,00:07:39,NaN,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
2,SANTOS,2022-01-16,2022-01-15 00:00:00,19:30:27,NaN,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022
3,SANTOS,2022-01-11,NaN,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
4,SANTOS,2022-02-05,2021-10-17 00:00:00,NaN,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022


# SET PERIOD OF THE DAY

In [48]:
# função de definição do periodo do dia
# period of the day definition function

def set_period(value: str) -> str:
    """
    PT-BR: recebe um horário em string, converte para
           time.time() e retorna um perído com base no horário.

    EN: recives a time as a format, converts it to time.time()
        and returns a period of the day based on the time.

    params
    ----------
    value : str

        PT-BR: horário em string.
        EN: time as a string format.

    return:
    ----------
    str
        horário convertido em período.
        time converted into a period of the day.
    """

    # verifica se é do tipo time.time()
    # check if it is time.time() type
    if isinstance(value, time):
        time_occ = value
    
    # se nao for, converte
    # if not, converts it
    else:
        try:
            time_occ = datetime.strptime(str(value), '%H:%M:%S').time()
        except ValueError:
            return 'Horário inválido'
        

    # categorizando os periodos do dia
    # categorizing the period of the day
    if time(0, 0, 0) <= time_occ <= time(5, 59, 59):        # MADRUGADA, entre 0h e 5h 59min 59s
        return 'DE MADRUGADA'                               # MADRUGADA (early morning), between 0am and 5am 59min 59s
    
    elif time(6, 0, 0) <= time_occ <= time(11, 59, 59):     # DE MANHÃ, entre 6h e 11h 59min 59s
        return 'DE MANHÃ'                                   # DE MANHÃ (morning), between 6am and 11am 59min 59s
    
    elif time(12, 0, 0) <= time_occ <= time(17, 59, 59):    # À TARDE, entre 12h e 17h 59min 59s
        return 'À TARDE'                                    # À TARDE (afternoon), between 12pm and 5pm 59min 59s
    
    elif time(18, 0, 0) <= time_occ <= time(23, 59, 59):    # À NOITE, entre 18h e 23h 59min 59s
        return 'À NOITE'                                    # À NOITE (night), between 6pm and 11pm 59min 59s
    

    # retorna horario invalido caso algum horario nao tenha sido inserido corretamente 
    # returns invalid time if any time occurence has been inputed incorrectly
    else:
        return 'Horário inválido'

In [49]:
test = set_period("19:30:27")
test

'À NOITE'

In [50]:
# aplicando a conversão de hora para período na coluna 'DESC_PERIODO'
# apllying the time conversion to period of the day in the 'DESC_PERIODO' column

df['DESC_PERIODO'] = df.apply(
    lambda row: set_period(row['HORA_OCORRENCIA_BO'])

    # se o valor da coluna de periodos for nulo, aplica a função de definição de períodos nela
    # if the period of the day column value is null, applies the period of the day definition funtion in it
    if pd.isna(row['DESC_PERIODO']) or row['DESC_PERIODO'] == ''

    # senão, deixa o valor como está, mantendo o período registrado
    # if not, keep the values how it is, keeping the inputed period of the day 
    else row['DESC_PERIODO'],
    axis=1
)

# apaga a os horários porque não serão mais necessárias
# dropping the time column beacuse it is not more necessary
df_standard_time = df.drop(columns='HORA_OCORRENCIA_BO')
df_standard_time

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA
0,SANTOS,2022-01-05,2021-11-20 00:00:00,À NOITE,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022
1,SANTOS,2022-01-09,2022-01-09 00:00:00,DE MADRUGADA,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
2,SANTOS,2022-01-16,2022-01-15 00:00:00,À NOITE,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022
3,SANTOS,2022-01-11,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
4,SANTOS,2022-02-05,2021-10-17 00:00:00,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022
...,...,...,...,...,...,...,...,...,...
213,SANTOS,2024-05-08,2024-05-01,EM HORA INCERTA,CASA,SANTA MARIA,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024
214,SANTOS,2024-05-15,2024-05-05,À NOITE,CASAS,RADIO CLUBE,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024
215,SANTOS,2024-05-21,2024-05-21,DE MANHÃ,VIA PÚBLICA,GONZAGA,ART. 213 - ESTUPRO,5,2024
216,SANTOS,2024-05-21,2011-06-08,PELA MANHÃ,APARTAMENTO,JOSÉ MENINO,ART. 213 - ESTUPRO,5,2024


# SET NEIGHBOURHOOD

In [51]:
# normalizando os valores dos bairros
# normalizing the neighbourhood values

df_standard_time['BAIRRO'] = pd.DataFrame(unidecode(bairro.upper()) for bairro in df_standard_time['BAIRRO'])
df_standard_time['BAIRRO'].unique()

array(['VILA MATIAS', 'MARAPE', 'MACUCO', 'MORRO NOVA CINTRA', 'ESTUARIO',
       'APARECIDA', 'SANTA MARIA', 'GONZAGA', 'EMBARE', 'CHICO DE PAULA',
       'CASTELO', 'BOQUEIRAO', 'MORRO DE SAO BENTO', 'CENTRO', 'PAQUETA',
       'VILA SAO JORGE', 'AREA RURAL', 'PONTA DA PRAIA', 'CANELEIRA',
       'JOSE MENINO', 'ENCRUZILHADA', 'SAO MANOEL', 'VILA VOTURUA',
       'VILA NOVA', 'RADIO CLUBE', 'AREIA BRANCA', 'CARUARA',
       'CAMPO GRANDE', 'ALEMOA', 'MORRO DO PACHECO'], dtype=object)

In [52]:
# temos apenas dados de santos, então a coluna "NOMES_MUNICIPIO" não é mais necessária
# we only have santos data, so the "NOMES_MUNICIPIO" (the city column) is no longer necessary

df_new = df_standard_time.drop(columns='NOME_MUNICIPIO')
df_new

,DATA_REGISTRO,DATA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA
0,2022-01-05,2021-11-20 00:00:00,À NOITE,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022
1,2022-01-09,2022-01-09 00:00:00,DE MADRUGADA,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
2,2022-01-16,2022-01-15 00:00:00,À NOITE,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022
3,2022-01-11,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022
4,2022-02-05,2021-10-17 00:00:00,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022
...,...,...,...,...,...,...,...,...
213,2024-05-08,2024-05-01,EM HORA INCERTA,CASA,SANTA MARIA,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024
214,2024-05-15,2024-05-05,À NOITE,CASAS,RADIO CLUBE,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024
215,2024-05-21,2024-05-21,DE MANHÃ,VIA PÚBLICA,GONZAGA,ART. 213 - ESTUPRO,5,2024
216,2024-05-21,2011-06-08,PELA MANHÃ,APARTAMENTO,JOSE MENINO,ART. 213 - ESTUPRO,5,2024


# SET MONTH NAME

In [53]:
# nomes dos meses num dicionario de conversão
# month names in a conversion dict

months = {
    1: 'JANEIRO',
    2: 'FEVEREIRO',
    3: 'MARÇO',
    4: 'ABRIL',
    5: 'MAIO',
    6: 'JUNHO',
    7: 'JULHO',
    8: 'AGOSTO',
    9: 'SETEMBRO',
    10: 'OUTUBRO',
    11: 'NOVEMBRO',
    12: 'DEZEMBRO',
}

df_new['MES'] = [months.get(n, "INCERTO") for n in df['MES_ESTATISTICA']]
df_new['MES']

0        JANEIRO
1        JANEIRO
2        JANEIRO
3        JANEIRO
4      FEVEREIRO
         ...    
213         MAIO
214         MAIO
215         MAIO
216         MAIO
217        JUNHO
Name: MES, Length: 218, dtype: object

# DATES

In [54]:
print(
    "occorence date: ", df_new['DATA_OCORRENCIA_BO'][0],
    "\n"
    "report date: ", df_new['DATA_REGISTRO'][0]
)

occorence date:  2021-11-20 00:00:00 
report date:  2022-01-05


PT-BR: para saber quanto tempo uma vitima levou para registrar uma ocorrência, podemos usar uma diferenciação
de data. porém, é preciso que ambas as datas estejam no mesmo formato. visualiza-se que o atributo da data de
ocorrência apresenta inconsistências de formato, então é preciso tratar isso.

EN: to find out how long it took a victm to file a report, we can use a date differentiation. however, both of
date must be in the same format. we see that the date of occurence feature has format inconsistences, so it needs
to be addressed.

In [55]:
# normalizando o formato das colunas data
# normalizing the date columns' format

def only_date(date_value: str, format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    PT-BR: procedimento para extrair uma data de uma string num formato específico e a retornar no padrão ISO.
           ignora a data se já estiver no devido formato.

    EN: procediment to extract a date from a string in a especific format and return it like ISO standard.
        it ignores the date if it already is in the correct format.

    params
    ----------
    date_value : str

        PT-BR: data a ser formatada.
        EN: date to be formated.

    format : str

        PT-BR: formato específico da data de entrada. formato "%Y-%m-%d %H:%M:%S" como padrão.
        EN: input date's specified format. "%Y-%m-%d %H:%M:%S" as standard format.

    return:
    ----------
    str
        data formatada como %Y-%m-%d, padrão ISO.
        time formated like %Y-%m-%dm ISO standard.
    """
    
    if pd.isna(date_value):
        return None

    try:
        dt_format = datetime.strptime(date_value, format)
        dt = dt_format.date()

        return dt.isoformat()
    
    # considero que se o valor nao estiver no formato data + hora, então está no formato de apenas data
    # i am considering that if the value is not in date + hour format then it is in only date format
    except ValueError:
        return date_value


In [56]:
# data de ocorência normalizada
# occurrence date standardized

df_new['DATA_OCORRENCIA_BO'] = df_new['DATA_OCORRENCIA_BO'].apply(only_date)
df_new['DATA_OCORRENCIA_BO']

0      2021-11-20
1      2022-01-09
2      2022-01-15
3            None
4      2021-10-17
          ...    
213    2024-05-01
214    2024-05-05
215    2024-05-21
216    2011-06-08
217    2024-06-17
Name: DATA_OCORRENCIA_BO, Length: 218, dtype: object

In [57]:

week_day = []

for day in df_new['DATA_OCORRENCIA_BO']:
    if day == None:
        week_day.append(day)
        continue
    
    day = datetime.strptime(day, "%Y-%m-%d").weekday()
    week_day.append(day)

In [58]:
days_of_week = {
    0: "SEGUNDA", 
    1: "TERÇA",
    2: "QUARTA",
    3: "QUINTA",
    4: "SEXTA",
    5: "SABADO",
    6: "DOMINGO"
    }

df_new['DIA_SEMANA'] = [days_of_week.get(n, 'INCERTO') for n in week_day]
df_new['DIA_SEMANA']

0       SABADO
1      DOMINGO
2       SABADO
3      INCERTO
4      DOMINGO
        ...   
213     QUARTA
214    DOMINGO
215      TERÇA
216     QUARTA
217    SEGUNDA
Name: DIA_SEMANA, Length: 218, dtype: object

In [59]:
# diferença entre as datas
# dates difference

def diff_days(d1, d2):
    """
    PT-BR: procedimento para calcular a diferença em dias entre duas datas no formato "%Y-%m-%d".
           caso alguma das datas seja nula, retorna 'INCERTO'.

    EN: procedure to calculate the difference in days between two dates in "%Y-%m-%d" format.
        if any of the dates is null, returns 'INCERTO'.

    params
    ----------
    d1 : str
        PT-BR: primeira data no formato "%Y-%m-%d".
        EN: first date in "%Y-%m-%d" format.

    d2 : str
        PT-BR: segunda data no formato "%Y-%m-%d".
        EN: second date in "%Y-%m-%d" format.

    return:
    ----------
    int | str
        PT-BR: diferença em dias entre as duas datas ou 'INCERTO' caso algum valor seja nulo.
        EN: difference in days between both dates or 'INCERTO' if any value is null.
    """

    for date in [d1, d2]:
        
        if pd.isna(date):
            return "INCERTO"

    # datetime objects
    d1 = datetime.strptime(d1, "%Y-%m-%d")
    d2 = datetime.strptime(d2, "%Y-%m-%d")

    return int((d2 - d1).days)


In [60]:
diff_days("2021-11-20", "2022-01-05")

46

In [61]:
# coluna para a diferença de dias
# days diferrence column

df_new['DIFERENCA_DIAS'] = df_new.apply(
    lambda row: diff_days(row['DATA_OCORRENCIA_BO'], row['DATA_REGISTRO']),
    axis=1
    )

df_new['DIFERENCA_DIAS']

0           46
1            0
2            1
3      INCERTO
4          111
        ...   
213          7
214         10
215          0
216       4731
217          1
Name: DIFERENCA_DIAS, Length: 218, dtype: object

In [62]:
# lista e tabela de frequência para o intevalo de dias
# list and frequency table to the interval of days

delta = []

for i in df_new['DIFERENCA_DIAS']:
    if i != "INCERTO":
        delta.append(i)

delta_table = pd.Series(delta).value_counts()
delta_table

0       52
1       30
2       14
3        6
6        4
46       4
11       4
10       4
5        2
7057     2
113      2
614      2
35       2
37       2
2240     2
150      2
222      2
475      2
14       2
730      2
65       2
111      2
529      2
689      2
184      2
1827     2
4        2
8        2
41       2
2446     2
74       2
61       2
56       2
9        2
1013     2
441      2
1865     2
27       2
28       2
4738     2
1462     2
40       2
278      2
24       2
103      2
38       2
734      2
33       2
4052     2
439      2
68       2
13       2
7        2
4731     2
Name: count, dtype: int64

In [63]:
# para facilitar a contagem de ocorrêcnias e os gráficos
# to facilite the count of occurences and graphs

df_new['N'] = [1 for n in range(0, len(df_new))]
df_new['N']

0      1
1      1
2      1
3      1
4      1
      ..
213    1
214    1
215    1
216    1
217    1
Name: N, Length: 218, dtype: int64

# SAVE

In [64]:
'''
PT-BR: as culunas de datas foram úteis no processo de limpeza, de tratamento e de extração de novas
features, mas não serão mais relevantes para as visualizações no power bi, então elas serão
dropadas antes do carregamento.

EN: the date columns were useful in the process of cleaning, processing and extracting new features,
but they will no longer be relevant for visualizations in power bi, so they will be dropped before
loading.
'''

df_final = df_new.drop(columns=['DATA_REGISTRO', 'DATA_OCORRENCIA_BO'])

In [65]:
# dataframe
df_final

,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,MES,DIA_SEMANA,DIFERENCA_DIAS,N
0,À NOITE,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022,JANEIRO,SABADO,46,1
1,DE MADRUGADA,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,JANEIRO,DOMINGO,0,1
2,À NOITE,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022,JANEIRO,SABADO,1,1
3,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,JANEIRO,INCERTO,INCERTO,1
4,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022,FEVEREIRO,DOMINGO,111,1
...,...,...,...,...,...,...,...,...,...,...
213,EM HORA INCERTA,CASA,SANTA MARIA,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024,MAIO,QUARTA,7,1
214,À NOITE,CASAS,RADIO CLUBE,ESTUPRO DE VULNERAVEL (ART.217-A),5,2024,MAIO,DOMINGO,10,1
215,DE MANHÃ,VIA PÚBLICA,GONZAGA,ART. 213 - ESTUPRO,5,2024,MAIO,TERÇA,0,1
216,PELA MANHÃ,APARTAMENTO,JOSE MENINO,ART. 213 - ESTUPRO,5,2024,MAIO,QUARTA,4731,1


In [67]:
df_final.to_csv("C:\\Users\\SAMSUNG\\OneDrive\\Documentos\\PROJETO INTEGRADOR 1\\dados\\df", index=False)
delta_table.to_csv("C:\\Users\\SAMSUNG\\OneDrive\\Documentos\\PROJETO INTEGRADOR 1\\dados\\delta_table")